In [1]:
import os
import sys
torch_cuda_path = r"E:\PyTorchGPU\venv\Lib\site-packages\torch\lib"
os.environ['PATH'] = torch_cuda_path + os.pathsep + os.environ.get('PATH', '')
if sys.platform == 'win32' and hasattr(os, 'add_dll_directory'):
    os.add_dll_directory(torch_cuda_path)

print("CUDA PATH configured")
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print("GPU 1 (NVIDIA GeForce) selected")

print("="*70)

CUDA PATH configured
GPU 1 (NVIDIA GeForce) selected


In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import wfdb
from scipy import signal
from scipy.signal import find_peaks, butter, filtfilt, wiener
from scipy.spatial.distance import pdist, squareform
import pywt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, classification_report, 
                             roc_curve, auc)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
from tqdm.notebook import tqdm
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("\n" + "="*70)
print("PyTorch & CUDA Setup")
print("="*70)

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA is available!")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")
    
print(f"Device set to: {device}")
print("="*70)


PyTorch & CUDA Setup
CUDA not available. Using CPU.
Device set to: cpu


In [ ]:
class MITBIHDataLoader:
    def __init__(self, data_dir='mitbih_data', sampling_rate=360):
        self.data_dir = Path(data_dir)
        self.data_dir.mkdir(exist_ok=True)
        self.sampling_rate = sampling_rate
        self.record_ids = [
            '100', '101', '102', '103', '104', '105', '106', '107', '108', '109',
            '111', '112', '113', '114', '115', '116', '117', '118', '119', '121',
            '122', '123', '124', '200', '201', '202', '203', '205', '207', '208',
            '209', '210', '212', '213', '214', '215', '217', '219', '220', '221',
            '222', '223', '228', '230', '231', '232', '233', '234'
        ]
        self.aami_classes = {
            'N': ['N', 'L', 'R', 'e', 'j'],
            'S': ['A', 'a', 'J', 'S'],
            'V': ['V', 'E'],
            'F': ['F'],
            'Q': ['/', 'f', 'Q']
        }
    def download_record(self, record_id):
        try:
            record = wfdb.rdrecord(record_id, pn_dir='mitdb', sampfrom=0, sampto=None)
            annotation = wfdb.rdann(record_id, 'atr', pn_dir='mitdb')
            return record, annotation
        except Exception as e:
            print(f"Download error {record_id}: {e}")
            return None, None

    def preprocess_signal(self, signal, method='wiener'):
        if method == 'bandpass':
            nyquist = self.sampling_rate / 2
            low = 0.5 / nyquist
            high = 40.0 / nyquist

            b, a = butter(4, [low, high], btype='band')
            filtered = filtfilt(b, a, signal)

        elif method == 'wiener':
            filtered = wiener(signal, mysize=55)  # size = 45,65,75,95 is worst vai
        else:
            raise ValueError("Unknown method! Choose 'bandpass' or 'wiener'.")

        normalized = (filtered - np.mean(filtered)) / (np.std(filtered) + 1e-8)
        return normalized


    def segment_beats(self, signal, r_peaks, window_size=200):
        beats = []
        half_window = window_size // 2

        for peak in r_peaks:
            start = peak - half_window
            end = peak + half_window
            if start >= 0 and end < len(signal):
                beat = signal[start:end]
                if len(beat) == window_size:
                    beats.append(beat)
        return np.array(beats)

    def map_to_aami(self, annotation_symbol):
        for aami_class, symbols in self.aami_classes.items():
            if annotation_symbol in symbols:
                return aami_class
        return 'Q'

    def load_dataset(self, num_records=10, beat_window=360, binary_classification=True):
        all_beats = []
        all_labels = []

        print(f"Loading {num_records} MIT-BIH records...")
        print("=" * 70)

        for i, record_id in enumerate(self.record_ids[:num_records]):
            print(f"Loading {record_id}... ({i+1}/{num_records})", end='\r')

            record, annotation = self.download_record(record_id)
            if record is None:
                continue

            ecg_signal = record.p_signal[:, 0]
            ecg_clean = self.preprocess_signal(ecg_signal)

            r_peaks = annotation.sample
            beat_types = annotation.symbol

            beats = self.segment_beats(ecg_clean, r_peaks, window_size=beat_window)

            valid_indices = []
            half_window = beat_window // 2

            for idx, peak in enumerate(r_peaks):
                if peak - half_window >= 0 and peak + half_window < len(ecg_signal):
                    valid_indices.append(idx)

            labels = [beat_types[i] for i in valid_indices if i < len(beat_types)]
            labels = labels[:len(beats)]

            aami_labels = [self.map_to_aami(label) for label in labels]

            if binary_classification:
                binary_labels = [0 if label == 'N' else 1 for label in aami_labels]
                all_labels.extend(binary_labels)
            else:
                all_labels.extend(aami_labels)

            all_beats.extend(beats)

        print(f"\nLoaded {len(all_beats)} beats from {num_records} records")
        X = np.array(all_beats)
        y = np.array(all_labels)
        unique, counts = np.unique(y, return_counts=True)
        print("\n Class Distribution:")
        for cls, count in zip(unique, counts):
            print(f"  Class {cls}: {count} samples ({count/len(y)*100:.1f}%)")
        return X, y
loader = MITBIHDataLoader()


: 

In [ ]:
X, y = loader.load_dataset(
    num_records=48,
    beat_window=360,
    binary_classification=True
)

print("Dataset Summary:")
print(f"Total beats: {len(X)}")
print(f"Signal shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Signal length: {X.shape[1]} samples ({X.shape[1]/360:.2f} seconds)")
print(f"Class balance: {np.sum(y==0)} Normal, {np.sum(y==1)} Abnormal")


Loading 48 MIT-BIH records...


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Sample ECG Beats from MIT-BIH Database', fontsize=16, fontweight='bold')
normal_indices = np.where(y == 0)[0]
for i, ax in enumerate(axes[0]):
    idx = normal_indices[np.random.randint(0, len(normal_indices))]
    ax.plot(X[idx], linewidth=1.5, color='green')
    ax.set_title(f'Normal Beat {i+1}', fontweight='bold')
    ax.set_xlabel('Samples')
    ax.set_ylabel('Amplitude (normalized)')
    ax.grid(alpha=0.3)
abnormal_indices = np.where(y == 1)[0]
for i, ax in enumerate(axes[1]):
    idx = abnormal_indices[np.random.randint(0, len(abnormal_indices))]
    ax.plot(X[idx], linewidth=1.5, color='red')
    ax.set_title(f'Abnormal Beat {i+1}', fontweight='bold')
    ax.set_xlabel('Samples')
    ax.set_ylabel('Amplitude (normalized)')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
class AdvancedECGFeatures:
    def __init__(self, sampling_rate=360):
        self.fs = sampling_rate
    
    def extract_wavelet_features(self, signal, wavelet='db4', level=4):
        coeffs = pywt.wavedec(signal, wavelet, level=level)     
        features = {}
        cA = coeffs[0]
        features['wavelet_cA_mean'] = np.mean(np.abs(cA))
        features['wavelet_cA_std'] = np.std(cA)
        features['wavelet_cA_energy'] = np.sum(cA**2)
        for i, cD in enumerate(coeffs[1:], 1):
            features[f'wavelet_cD{i}_mean'] = np.mean(np.abs(cD))
            features[f'wavelet_cD{i}_std'] = np.std(cD)
            features[f'wavelet_cD{i}_energy'] = np.sum(cD**2)
        
        return features
    
    def extract_statistical_features(self, signal):
        features = {
            'mean': np.mean(signal),
            'std': np.std(signal),
            'var': np.var(signal),
            'min': np.min(signal),
            'max': np.max(signal),
            'skewness': self._skewness(signal),
            'kurtosis': self._kurtosis(signal),
            'rms': np.sqrt(np.mean(signal**2)),
            'peak_to_peak': np.ptp(signal)
        }
        return features
    def extract_frequency_features(self, signal):
        fft_vals = np.fft.fft(signal)
        fft_mag = np.abs(fft_vals[:len(fft_vals)//2])
        fft_freq = np.fft.fftfreq(len(signal), 1/self.fs)[:len(fft_vals)//2]
        
        features = {
            'dominant_freq': fft_freq[np.argmax(fft_mag)],
            'spectral_entropy': self._spectral_entropy(fft_mag),
            'spectral_centroid': np.sum(fft_freq * fft_mag) / (np.sum(fft_mag) + 1e-8),
            'spectral_rolloff': self._spectral_rolloff(fft_mag, fft_freq)
        }
        
        return features
    
    def extract_all_features(self, signal):
        features = {}
        features.update(self.extract_wavelet_features(signal))
        features.update(self.extract_statistical_features(signal))
        features.update(self.extract_frequency_features(signal))
        return features
    def _skewness(self, x):
        return np.mean(((x - np.mean(x)) / (np.std(x) + 1e-8))**3)
    def _kurtosis(self, x):
        return np.mean(((x - np.mean(x)) / (np.std(x) + 1e-8))**4)
    def _spectral_entropy(self, power_spectrum):
        ps_norm = power_spectrum / (np.sum(power_spectrum) + 1e-8)
        ps_norm = ps_norm[ps_norm > 0]
        return -np.sum(ps_norm * np.log2(ps_norm + 1e-8))  
    def _spectral_rolloff(self, power_spectrum, frequencies, threshold=0.85):
        cumsum = np.cumsum(power_spectrum)
        rolloff_idx = np.where(cumsum >= threshold * cumsum[-1])[0]
        return frequencies[rolloff_idx[0]] if len(rolloff_idx) > 0 else 0
feature_extractor = AdvancedECGFeatures(sampling_rate=360)

In [ ]:
print("Extracting advanced features from all beats...")
all_features = []
for i, beat in enumerate(tqdm(X, desc="Extracting features")):
    features = feature_extractor.extract_all_features(beat)
    all_features.append(features)
features_df = pd.DataFrame(all_features)
features_df['label'] = y
print(f"\nFeature extraction complete!")
print(f"Extracted {features_df.shape[1]-1} features")
print(f"Feature names: {list(features_df.columns[:10])}...")
display(features_df.head())

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"  Training set: {len(X_train)} samples")
print(f"  Test set: {len(X_test)} samples")
print(f"\n  Train - Normal: {np.sum(y_train==0)}, Abnormal: {np.sum(y_train==1)}")
print(f"  Test  - Normal: {np.sum(y_test==0)}, Abnormal: {np.sum(y_test==1)}")

In [ ]:
class ECGDataset(Dataset): 
    def __init__(self, signals, labels):
        self.signals = torch.FloatTensor(signals).unsqueeze(1) 
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.signals)
    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

train_dataset = ECGDataset(X_train, y_train)
test_dataset = ECGDataset(X_test, y_test)
train_size = int(0.85 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size]
)
batch_size = 32
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
class AttentionModule(nn.Module):
    def __init__(self, in_channels):
        super(AttentionModule, self).__init__()
        self.query = nn.Conv1d(in_channels, in_channels // 8, 1)
        self.key = nn.Conv1d(in_channels, in_channels // 8, 1)
        self.value = nn.Conv1d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        B, C, L = x.size()
        proj_query = self.query(x).view(B, -1, L).permute(0, 2, 1)
        proj_key = self.key(x).view(B, -1, L)
        proj_value = self.value(x).view(B, -1, L)
        attention = torch.bmm(proj_query, proj_key)
        attention = F.softmax(attention, dim=-1)
        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(B, C, L)
        out = self.gamma * out + x
        
        return out, attention


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, 
                              stride=stride, padding=kernel_size//2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, 
                              stride=1, padding=kernel_size//2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.2)
        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(residual)
        out = self.relu(out)
        return out


class ECG_CNN_Attention(nn.Module):
    def __init__(self, input_length=360, num_classes=2):
        super(ECG_CNN_Attention, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.res_block1 = ResidualBlock(64, 128, stride=2)
        self.res_block2 = ResidualBlock(128, 256, stride=2)
        self.res_block3 = ResidualBlock(256, 512, stride=2)
        self.res_block4 = ResidualBlock(512, 512, stride=1)
        self.attention = AttentionModule(512)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
        self.attention_weights = None
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.res_block1(x)
        x = self.res_block2(x)
        x = self.res_block3(x)
        x = self.res_block4(x)
        x, self.attention_weights = self.attention(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x
    
    def get_attention_map(self):
        return self.attention_weights

model = ECG_CNN_Attention(input_length=360, num_classes=2).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: {total_params*4/1024/1024:.2f} MB")

In [ ]:
epochs = 50
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                 factor=0.5, patience=5)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}
print(f"  Epochs: {epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Optimizer: Adam")
print(f"  Scheduler: ReduceLROnPlateau")

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))  


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for signals, labels in dataloader:
        signals, labels = signals.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(signals)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for signals, labels in dataloader:
            signals, labels = signals.to(device), labels.to(device)
            
            outputs = model(signals)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc


print("Training")
print("=" * 70)
best_val_acc = 0.0
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn_attention_model.pth')
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        print('-' * 70)

print(f"\nTraining Complete!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(history['train_loss'], label='Training Loss', linewidth=2, marker='o', markersize=4)
ax1.plot(history['val_loss'], label='Validation Loss', linewidth=2, marker='s', markersize=4)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Learning Curves - Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax2.plot(history['train_acc'], label='Training Accuracy', linewidth=2, marker='o', markersize=4)
ax2.plot(history['val_acc'], label='Validation Accuracy', linewidth=2, marker='s', markersize=4)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Learning Curves - Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
model.load_state_dict(torch.load('best_cnn_attention_model.pth'))
model.eval()

all_predictions = []
all_labels = []
all_probabilities = []

with torch.no_grad():
    for signals, labels in test_loader:
        signals = signals.to(device)
        outputs = model(signals)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

y_pred = np.array(all_predictions)
y_true = np.array(all_labels)
y_prob = np.array(all_probabilities)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='binary')
recall = recall_score(y_true, y_pred, average='binary')
f1 = f1_score(y_true, y_pred, average='binary')

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

auc_roc = roc_auc_score(y_true, y_prob[:, 1])

print(" " * 70)
print("TEST SET PERFORMANCE")
print(" " * 70)
print(f"Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1-Score:    {f1:.4f}")
print(f"AUC-ROC:     {auc_roc:.4f}")
print(" " * 70)

print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal']))

fig, ax = plt.subplots(figsize=(10, 8))  # figure size also slightly increased

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Abnormal'],
            yticklabels=['Normal', 'Abnormal'],
            cbar_kws={'label': 'Count'},
            annot_kws={'size': 28, 'weight': 'bold'},  # number size inside boxes
            ax=ax)

ax.set_xlabel('Predicted Label', fontsize=18, fontweight='bold', labelpad=12)
ax.set_ylabel('True Label', fontsize=18, fontweight='bold', labelpad=12)
ax.set_title('Confusion Matrix', fontsize=22, fontweight='bold', pad=15)

ax.set_xticklabels(['Normal', 'Abnormal'], fontsize=16)
ax.set_yticklabels(['Normal', 'Abnormal'], fontsize=16, rotation=0)

# colorbar label size
ax.collections[0].colorbar.ax.tick_params(labelsize=13)
ax.collections[0].colorbar.set_label('Count', fontsize=14)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

fpr, tpr, thresholds = roc_curve(y_true, y_prob[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, linewidth=3, label=f'AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], linewidth=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("Feature Importance Analysis")
print("=" * 70)

test_features = []
for beat in X_test[:100]:
    features = feature_extractor.extract_all_features(beat)
    test_features.append(features)

features_test_df = pd.DataFrame(test_features)

correlations = features_test_df.corrwith(pd.Series(y_test[:100]))
top_features = correlations.abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
top_features.plot(kind='barh', color='steelblue')
plt.xlabel('Absolute Correlation with Label')
plt.ylabel('Feature')
plt.title('Top 15 Most Important Features')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Feature importance analysis complete.")
print("\nTop 5 Features:")
for i, (feat, corr) in enumerate(top_features.head().items(), 1):
    print(f"{i}. {feat}: {abs(corr):.4f}")

results = {
    'model_name': 'CNN_Attention',
    'test_accuracy': float(accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_specificity': float(specificity),
    'test_f1': float(f1),
    'test_auc_roc': float(auc_roc),
    'confusion_matrix': cm.tolist(),
    'training_history': history
}

import json
with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to results.json")

results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'AUC-ROC'],
    'Value': [accuracy, precision, recall, specificity, f1, auc_roc]
})

display(results_df.style.format({'Value': '{:.4f}'}))

results_df.to_csv('results_summary.csv', index=False)
print("Results summary saved to CSV")


In [ ]:
table = pd.DataFrame({
    'Metric': [
        'Accuracy (%)',
        'Precision',
        'Recall (Sensitivity)',
        'Specificity',
        'F1-Score',
        'AUC-ROC',
        'True Positives',
        'True Negatives',
        'False Positives',
        'False Negatives'
    ],
    'Value': [
        f'{accuracy*100:.2f}',
        f'{precision:.4f}',
        f'{recall:.4f}',
        f'{specificity:.4f}',
        f'{f1:.4f}',
        f'{auc_roc:.4f}',
        f'{tp}',
        f'{tn}',
        f'{fp}',
        f'{fn}'
    ]
})
print(" " * 70)
print("TABLE")
print(" " * 70)
display(table)

latex_table = table.to_latex(index=False, caption='Performance Metrics on MIT-BIH Test Set', 
                                        label='tab:results')

with open('results_table.tex', 'w') as f:
    f.write(latex_table)

print("LaTeX table saved to results_table.tex")

class BaselineCNN(nn.Module):
    def __init__(self, input_length=360, num_classes=2):
        super(BaselineCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7, stride=2, padding=3)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=5, stride=2, padding=2)
        self.bn3 = nn.BatchNorm1d(256)
        self.conv4 = nn.Conv1d(256, 512, kernel_size=3, stride=2, padding=1)
        self.bn4 = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
   
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

# CNN + Residual (No Attention)
class CNN_WithResidual(nn.Module):
    def __init__(self, input_length=360, num_classes=2):
        super(CNN_WithResidual, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.res_block1 = ResidualBlock(64, 128, stride=2)
        self.res_block2 = ResidualBlock(128, 256, stride=2)
        self.res_block3 = ResidualBlock(256, 512, stride=2)
        self.res_block4 = ResidualBlock(512, 512, stride=1)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.res_block1(x)
        x = self.res_block2(x)
        x = self.res_block3(x)
        x = self.res_block4(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

# CNN + Attention (Shallow)
class CNN_WithAttention(nn.Module):
    def __init__(self, input_length=360, num_classes=2):
        super(CNN_WithAttention, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7, stride=2, padding=3)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=5, stride=2, padding=2)
        self.bn3 = nn.BatchNorm1d(256)
        self.conv4 = nn.Conv1d(256, 512, kernel_size=3, stride=2, padding=1)
        self.bn4 = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.attention = AttentionModule(512)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x, _ = self.attention(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

print("Model variants defined!")

def quick_train(model, train_loader, val_loader, device, epochs=20, name="Model"):
    print(f"\nTraining {name}...")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    best_acc = 0
    best_state = None
    
    for epoch in range(epochs):
        model.train()
        for signals, labels in train_loader:
            signals, labels = signals.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(signals)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for signals, labels in val_loader:
                signals, labels = signals.to(device), labels.to(device)
                outputs = model(signals)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_acc = 100 * correct / total
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict().copy()
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Val Acc = {val_acc:.2f}%")
    
    model.load_state_dict(best_state)
    print(f"✓ {name} done! Best Val Acc: {best_acc:.2f}%")
    return model

def evaluate(model, test_loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for signals, labels in test_loader:
            signals = signals.to(device)
            outputs = model(signals)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds) * 100
    sensitivity = recall_score(all_labels, all_preds) * 100
    cm = confusion_matrix(all_labels, all_preds)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) * 100
    auc = roc_auc_score(all_labels, all_probs[:, 1])
    
    return {
        'Accuracy': accuracy,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'AUC': auc
    }

print("✓ Functions ready!")
```

---

## 🔹 STEP 3: Run Ablation Study (1টা cell - এটা সময় নেবে ~30 min)

```python
# ============== Cell 3: Run Ablation Study ==============

print("="*70)
print("STARTING ABLATION STUDY")
print("="*70)

results = []

# 1. Baseline CNN
print("\n[1/4] Baseline CNN...")
baseline = BaselineCNN().to(device)
baseline = quick_train(baseline, train_loader, val_loader, device, epochs=20, name="Baseline CNN")
metrics = evaluate(baseline, test_loader, device)
results.append({'Configuration': 'Baseline CNN', **metrics})
print(f"   Test: Acc={metrics['Accuracy']:.2f}%, AUC={metrics['AUC']:.4f}")
del baseline
torch.cuda.empty_cache()

# 2. CNN + Residual
print("\n[2/4] CNN + Residual...")
residual = CNN_WithResidual().to(device)
residual = quick_train(residual, train_loader, val_loader, device, epochs=20, name="CNN + Residual")
metrics = evaluate(residual, test_loader, device)
results.append({'Configuration': 'CNN + Residual', **metrics})
print(f"   Test: Acc={metrics['Accuracy']:.2f}%, AUC={metrics['AUC']:.4f}")
del residual
torch.cuda.empty_cache()

# 3. CNN + Attention (Shallow)
print("\n[3/4] CNN + Attention...")
attention = CNN_WithAttention().to(device)
attention = quick_train(attention, train_loader, val_loader, device, epochs=20, name="CNN + Attention")
metrics = evaluate(attention, test_loader, device)
results.append({'Configuration': 'CNN + Attention (Shallow)', **metrics})
print(f"   Test: Acc={metrics['Accuracy']:.2f}%, AUC={metrics['AUC']:.4f}")
del attention
torch.cuda.empty_cache()
print("\n[4/4] Full Model...")
model.load_state_dict(torch.load('best_cnn_attention_model.pth'))
metrics = evaluate(model, test_loader, device)
results.append({'Configuration': 'Full Model (Res + Attn)', **metrics})
print(f"   Test: Acc={metrics['Accuracy']:.2f}%, AUC={metrics['AUC']:.4f}")
df = pd.DataFrame(results)
print("\n" + "="*70)
print("RESULTS")
print("="*70)
print(df.to_string(index=False))
df.to_csv('ablation_results.csv', index=False)
print("\n Saved to ablation_results.csv")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#95a5a6', '#3498db', '#e67e22', '#2ecc71']

for idx, (ax, metric) in enumerate(zip(axes, ['Accuracy', 'Sensitivity', 'AUC'])):
    data = df[metric]
    bars = ax.barh(df['Configuration'], data, color=colors, edgecolor='black', linewidth=2)
    bars[-1].set_linewidth(3)  # Highlight full model
    
    ax.set_xlabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} Comparison', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for bar in bars:
        width = bar.get_width()
        label = f'{width:.4f}' if metric == 'AUC' else f'{width:.2f}%'
        ax.text(width + 0.3, bar.get_y() + bar.get_height()/2, 
                label, va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('ablation_comparison.png', dpi=300, bbox_inches='tight')
print("Saved: ablation_comparison.png")
plt.show()

baseline_acc = df.iloc[0]['Accuracy']
improvements = df['Accuracy'] - baseline_acc
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(df)), improvements, color=colors, edgecolor='black', linewidth=2)
ax.set_xticks(range(len(df)))
ax.set_xticklabels(['Baseline', 'Residual', 'Attention', 'Full'])
ax.set_ylabel('Improvement (%)', fontsize=12, fontweight='bold')
ax.set_title('Component Contribution', fontsize=13, fontweight='bold')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    label = f'+{height:.2f}%' if height > 0 else 'Baseline' if height == 0 else f'{height:.2f}%'
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.1,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('ablation_contribution.png', dpi=300, bbox_inches='tight')
print("Saved: ablation_contribution.png")
plt.show()

print("\nABLATION STUDY COMPLETE!")
print("\nLaTeX TABLE FOR PAPER:")
print("="*70 + "\n")
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Ablation study results}")
print(r"\label{tab:ablation}")
print(r"\begin{tabular}{lccc}")
print(r"\hline")
print(r"\textbf{Configuration} & \textbf{Acc.} & \textbf{Sen.} & \textbf{AUC} \\")
print(r"\hline")
for i, row in df.iterrows():
    if i == len(df) - 1:
        print(f"\\textbf{{{row['Configuration']:30s}}} & "
              f"\\textbf{{{row['Accuracy']:.2f}}} & "
              f"\\textbf{{{row['Sensitivity']:.2f}}} & "
              f"\\textbf{{{row['AUC']:.3f}}} \\\\")
    else:
        print(f"{row['Configuration']:30s} & "
              f"{row['Accuracy']:.2f} & "
              f"{row['Sensitivity']:.2f} & "
              f"{row['AUC']:.3f} \\\\")

print(r"\hline")
print(r"\end{tabular}")
print(r"\end{table}")
print("\n" + "="*70)
print("Copy the above table into your LaTeX document!")
